# When do you need an agent, when do you need several, and what must an agent remember?

Question: on the same tasks, what does an agent buy over rules and over one LLM call? How much does each way of building the context cost, and what does each way forget? Which memory store answers right after a fact changes? When does a team of agents beat one agent that has the same token budget? Which single change helps an agent most, measured on items it has not seen?

Setup: a CPU notebook. It calls the NVIDIA-hosted, OpenAI-compatible API (`https://integrate.api.nvidia.com/v1`) with the Python standard library, pandas, numpy and scipy. It needs an `NVIDIA_API_KEY` secret. It uses `nvidia/nemotron-3.5-bolt-e4b` as the agent model of parts A to D (reasoning off, no rate limit seen), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` as the model upgrade of part E and the model of part F, and `nvidia/nemotron-3-ultra-550b-a55b` as the error reader. Nano-omni 30B and ultra allow about 40 requests per minute, so the code waits 2.5 s between two requests to the same model. Reasoning is off, except in one config of part A0. At most 4 requests run at the same time. The notebook stops at 2,900 HTTP attempts. Every call and token is counted from the API `usage` field. The API gives different answers between runs, even at temperature 0, so every part runs twice and both runs are shown. Part D has 12 tasks per family. Part F installs `nvidia-nat` 1.9.0 in its own `uv` virtual environment (Python 3.12), so it cannot change this kernel.

Exam objectives: agentic.1 (agents, rule-based systems, building blocks, harness, runtime), agentic.4 (feedback, prompt and workflow refinement, model selection), agentic.8 (NVIDIA NeMo, NeMo Agent Toolkit, NemoClaw), agentic.10 (stateless and stateful, context engineering), agentic.11 (multi-agent components, patterns, single against multi), agentic.14 (short-term and long-term memory).

Sources for NVIDIA product text: `ai-agents.txt`, `introduction-to-llm-agents.txt`, `multi-agent-systems.txt`, `agentic-inference.txt` and `building-nvidia-nemotron-3-agents-for-reasoning-multimodal-rag-voice-and-safety.txt` in `hub/sources`. Part F also follows the NeMo Agent Toolkit 1.9 documentation pages on the quick start, workflow configuration, the ReAct agent, evaluation and profiling.

Terms used below:

- **Generative AI call**: one request and one reply. The model cannot look anything up.
- **Rule-based system**: code with fixed patterns and fixed answers. It does only what its author wrote.
- **Agent**: an LLM that works in a loop. It picks a tool, reads the result, and picks again until it answers.
- **Tool call**: the model names a function and gives JSON arguments. Your code runs the function.
- **Harness**: the code around the model that runs tool calls and enforces limits. **Runtime**: the place where the agent runs, with limits on time, calls and tokens.
- **Stateless**: each request stands alone. **Stateful**: the system keeps state between turns.
- **Context window**: the text the model reads in one call. **Context engineering**: choosing what goes into that window. **Context rot**: facts get lost when the window holds a lot of irrelevant text.
- **Short-term memory**: what the agent holds in the current session. **Long-term memory**: a store outside the context that lasts across sessions. **BM25**: a keyword score that ranks stored text for a query.
- **Multi-agent system**: several agents with roles. A **supervisor** splits the task. A **pipeline** passes work from one agent to the next. A **generator and critic** pair checks each answer.
- **Token budget**: the most tokens (prompt plus completion, all calls) that one task may use.
- **Feedback loop**: measure, read the errors, change one thing, measure again. **Dev set**: items you tune on. **Held-out set**: items you only measure on.
- **Wilson interval**: the 95% range for a rate. **Exact McNemar test**: a paired test on the items where two systems disagree.


## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Part | What you measure | Your guess |
|---|---|---|
| A0 | Does the model need reasoning on to call a tool? Compare a small model and a larger one, reasoning off and on. | |
| A | Success of rules, one LLM call and an agent on 24 tasks. Which task kinds does each pass? | |
| A | How many calls and tokens does the agent need per task? | |
| B | Which context strategy keeps the early facts? Which one forgets them? | |
| B | What does a noisy conversation do to the full history? To the state object? | |
| C | Which store answers right after a fact changed? Which store gives stale answers? | |
| C | What happens when the store has no user scope? | |
| D | For which task family does a team of agents beat one agent at the same token budget? | |
| D | Which design uses the most tokens? Which one is the slowest? | |
| E | Which single change raises the dev score most? Does the held-out score follow? | |
| F | Does the NeMo Agent Toolkit install and run on Kaggle? What does its profiler show? | |


In [ ]:
# 1. Setup: constants, API key, and the part() runner. Standard library, pandas, numpy, scipy. Nothing to install.
import copy, http.client, json, math, os, random, re, subprocess, sys, textwrap, threading, time, traceback, urllib.error, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from scipy.stats import binomtest
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = os.environ.get("NIM_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL = "nvidia/nemotron-3.5-bolt-e4b"            # the model of every agent in parts A to D, reasoning off. It is not rate limited on this API.
UP = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"      # the model upgrade of part E, the model of part F, and a config of part A0. About 40 requests per minute.
BIG = "nvidia/nemotron-3-ultra-550b-a55b"         # reads the errors in part E and writes the better prompt. About 40 requests per minute.
SEED = 7
REPS_BY = dict(A=2, A0=2, B=2, C=2, D=2, E=2)     # runs per part. The API is not deterministic at temperature 0, so every part runs twice.
PACE_S, PACE_MODEL_S = 2.5, 0.4                   # seconds between two requests to the same model (PACE_MODEL_S for MODEL). On 2026-11-25 MODEL answered 429 after about 1,000 calls in one Kaggle session.
RUN_PARTS = "A0 A B C E D F".split()               # run some parts in a fresh session when the API limit stops a full run
CALL_CAP, MAX_INFLIGHT = 2900, 4                  # HTTP attempts of this kernel, without the calls of part F. At most 4 requests in flight.
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S, RETRY_CAP_S = 60, 5, 2.0, 40.0
PART_BUDGET_S = dict(A=180, A0=600, B=420, C=300, D=1080, E=480, F=600)   # seconds per part. The sum is 59 minutes. The notebook ends at 68.
T0 = time.time(); NOTEBOOK_END = T0 + 68 * 60; PART_END = NOTEBOOK_END
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
pd.set_option("display.width", 190); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 100)

def get_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        key = os.environ.get("NVIDIA_API_KEY")
    if not key:
        raise RuntimeError("NVIDIA_API_KEY not found. Add-ons > Secrets > add NVIDIA_API_KEY, then switch it on for this notebook.")
    return key

KEY = get_key()
print("key found:", bool(KEY))                       # never print the key itself
def scrub(s): return str(s).replace(KEY, "***")        # every error text goes through this before it is printed

STATUS, RES = [], {}                                  # status of each part, and the result tables of each part
def part(name, fn):
    """Run one part. On an exception: print the traceback, record the failure, and go on to the next part."""
    global PART_END
    if name not in RUN_PARTS: STATUS.append(dict(part=name, status="skipped", seconds=0, calls=0, note="not in RUN_PARTS")); return
    PART_END = min(time.time() + PART_BUDGET_S[name], NOTEBOOK_END); CUR["part"] = name
    t, a0 = time.time(), STATE["attempts"]
    try:
        fn(); st, note = "ok", ""
    except Exception:
        tb = scrub(traceback.format_exc()); print(tb[-1500:]); st, note = "FAILED", tb.strip().splitlines()[-1][:90]
    STATUS.append(dict(part=name, status=st, seconds=round(time.time() - t), calls=STATE["attempts"] - a0, note=note))

def show(df, title, name):
    print(f"\n{title}"); print(df.to_string(index=False)); df.to_csv(f"{OUT_DIR}/agents_{name}.csv", index=False)


In [ ]:
# 2. API helpers: one chat call with retry, a ledger of every HTTP attempt, at most 4 requests in flight, parallel map
LOCK, SEM = threading.Lock(), threading.BoundedSemaphore(MAX_INFLIGHT)
STATE = dict(attempts=0, ok=0, aborted=False)
PACE, PACE_LOCK = {}, threading.Lock()
LEDGER, CUR = [], dict(part="-")        # one ledger row per HTTP attempt. Every token count comes from the API usage field.

def http_post(url, body, headers, timeout):
    req = urllib.request.Request(url, body, headers)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read(), None
    except urllib.error.HTTPError as e:
        return e.code, e.read()[:200], e.headers.get("Retry-After")

def pace(model, deadline):
    """Keep a gap between two requests to one model. Returns False if the wait would pass the deadline."""
    gap = PACE_MODEL_S if model == MODEL else PACE_S
    with PACE_LOCK:
        now = time.time(); t = max(now, PACE.get(model, 0.0))
        if deadline and t >= deadline: return False
        PACE[model] = t + gap
    if t > now: time.sleep(t - now)
    return True

def chat(model, messages, tag, tools=None, max_tokens=300, deadline=None, think=False):
    """One model call. Retries 429, 5xx and network errors. Reasoning is off unless think=True. Returns a dict, never raises."""
    body = {"model": model, "messages": messages, "max_tokens": max_tokens, "temperature": 0.0, "seed": SEED,
            "chat_template_kwargs": {"enable_thinking": think}}
    if tools:
        body["tools"] = tools
    data, hdr = json.dumps(body).encode(), {"Authorization": "Bearer " + KEY, "Content-Type": "application/json"}
    out = dict(ok=False, error="", content="", tool_calls=[], finish="", ptok=0, ctok=0)
    for attempt in range(MAX_RETRIES + 1):
        with LOCK:
            if STATE["aborted"] or STATE["attempts"] >= CALL_CAP:
                out["error"] = "aborted" if STATE["aborted"] else "call cap"
                return out
            STATE["attempts"] += 1
        if not pace(model, deadline):
            with LOCK: STATE["attempts"] -= 1
            out["error"] = "deadline"; return out
        rec, t0 = dict(part=CUR["part"], tag=tag, model=model, status=0, ptok=0, ctok=0, lat=0.0), time.time()
        try:
            with SEM:
                status, raw, ra = http_post(BASE + "/chat/completions", data, hdr, REQ_TIMEOUT_S)
        except (OSError, http.client.HTTPException) as e:
            status, raw, ra = 0, b"", None; out["error"] = type(e).__name__
        rec["status"], rec["lat"] = status, round(time.time() - t0, 3)
        if status == 200:
            try:
                j = json.loads(raw); ch = j["choices"][0]; m = ch["message"]; u = j.get("usage") or {}
                rec.update(ptok=u.get("prompt_tokens") or 0, ctok=u.get("completion_tokens") or 0)
                out.update(ok=True, error="", content=m.get("content") or "", tool_calls=m.get("tool_calls") or [],
                           finish=ch.get("finish_reason") or "", ptok=rec["ptok"], ctok=rec["ctok"])
                with LOCK:
                    STATE["ok"] += 1
            except (ValueError, KeyError, IndexError, TypeError, AttributeError) as e:
                out["error"] = "bad reply " + type(e).__name__
            LEDGER.append(rec)
            return out
        LEDGER.append(rec)
        out["error"] = out["error"] or f"HTTP {status}"
        with LOCK:
            if STATE["ok"] == 0 and STATE["attempts"] >= 5:      # the first 5 calls all failed: stop the whole notebook early
                STATE["aborted"] = True
        if status and status != 429 and status < 500:             # other client errors are final
            return out
        try:
            wait = float(ra)
        except (TypeError, ValueError):
            wait = BACKOFF_S * 2 ** attempt
        wait = min(wait, RETRY_CAP_S)
        if attempt == MAX_RETRIES or (deadline and time.time() + wait >= deadline):
            break
        time.sleep(wait)
    return out

def pmap(fn, items, deadline, workers=MAX_INFLIGHT):
    """Run fn on every item. An item not started by the deadline (or after an abort) gives None."""
    def one(x):
        return fn(x) if time.time() < deadline and not STATE["aborted"] else None
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(one, items))


In [ ]:
# 3. Shared code: intervals, paired test, outcomes, summary table, and the agent loop. Pure Python.
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, (c - h) / d), min(1.0, (c + h) / d)

def rate_ci(k, n):
    if n == 0: return "n/a"
    lo, hi = wilson(k, n); return f"{100 * k / n:.0f} [{100 * lo:.0f},{100 * hi:.0f}]"

def maj(xs): return max(Counter(map(str, xs)).values()) / len(xs)      # baseline: always give the most common answer

def mcnemar(a, b):
    """Exact McNemar test on two lists of True/False for the same items. Returns (only a right, only b right, p)."""
    x, y = sum(i and not j for i, j in zip(a, b)), sum(j and not i for i, j in zip(a, b))
    return x, y, (1.0 if x + y == 0 else binomtest(x, x + y, 0.5).pvalue)

def paired(df, key, col, x, y, ok="ok"):
    a, b = df[df[col] == x].set_index(key)[ok], df[df[col] == y].set_index(key)[ok]
    idx = a.index.intersection(b.index); ox, oy, p = mcnemar(list(a[idx]), list(b[idx]))
    return dict(a=x, b=y, pairs=len(idx), only_a=ox, only_b=oy, p=round(p, 4))

def outcome(r, verdict):
    """Name the outcome of one reply. verdict: True, False, or None when the reply could not be parsed."""
    if not r["ok"]: return "error"
    if r["finish"] == "length": return "cut"
    if not r["content"].strip(): return "empty"
    return "unparsed" if verdict is None else ("ok" if verdict else "wrong")

def fin(r):
    r["ok"] = r["outcome"] == "ok"; r.setdefault("hits", int(r["ok"])); r.setdefault("items", 1); return r
def nr(**kw): return fin(dict(outcome="not_reached", calls=0, ptok=0, ctok=0, wall_s=0.0, **kw))
def rows_of(jobs, results, mk):    # None means the part deadline came first. That is an outcome, so it stays in the table.
    return [r if r is not None else mk(j) for j, r in zip(jobs, results)]

ORDER = dict(cond=["rules", "one_call", "agent"], kind=["exact_same", "exact_changed", "reworded", "multistep", "policy", "arithmetic"], var=["clean", "noisy"],
             strat=["stateless", "full_history", "sliding_window", "summary", "state_object"], store=["none", "dump", "bm25", "kv", "bm25_shared", "kv_shared"],
             family=["separable", "coupled", "simple"], system=["single", "supervisor_parallel", "pipeline", "generator_critic"],
             qkind=["same", "changed", "stable"], conf=["bolt_off", "omni30b_off", "omni30b_on"], cfg=["c0_baseline", "c1_check_step", "c2_prompt", "c3_model", "final"], split=["dev", "test"])

def say(text): print(textwrap.fill(text, 190))        # one paragraph, no line over 190 characters

def summ(df, by, specs=(("success", "hits", "items"),), means=()):
    """One row per group: pooled rate with Wilson 95% interval, the rate of each repeat, mean cost, and every outcome that is not ok."""
    rows = []
    df = df.assign(**{c: pd.Categorical(df[c], ORDER[c], ordered=True) for c in by if c in ORDER and set(df[c]) <= set(ORDER[c])})
    for key, d in df.groupby(by, observed=True):
        r = dict(zip(by, key if isinstance(key, tuple) else (key,))); r["n"] = len(d)
        for name, kc, nc in specs:
            r[name] = rate_ci(d[kc].sum(), d[nc].sum())
            for rep, dd in d.groupby("rep"):
                r[f"{name[:3]}_r{rep}"] = round(100 * dd[kc].sum() / dd[nc].sum())
        r.update(calls=round(d.calls.mean(), 1), tok=round((d.ptok + d.ctok).mean()), wall_s=round(d.wall_s.mean(), 1))
        r.update({m: round(d[m].mean(), 2) for m in means})
        r["other"] = " ".join(f"{k}:{v}" for k, v in d.outcome.value_counts().items() if k != "ok")
        rows.append(r)
    return pd.DataFrame(rows)

def samples(df):
    """Why replies were not ok: up to 3 sample replies for each outcome, 120 characters each. Fixes the blind spot of the outcome counts."""
    print("\nSample replies behind each outcome that is not ok (a cut or unparsed reply can hide a refusal such as 'I need the order ID'):")
    for oc, d in df[~df.outcome.isin(["ok", "not_reached"])].groupby("outcome"):
        for _, r in d.head(3).iterrows():
            print(f"  {oc:9s} {str(r.get('reply') or r.get('err') or '')[:120]!r}")

def jparse(text):
    """The first JSON object in a reply, or None."""
    text = text.split("</think>")[-1]
    try:
        j = json.loads(text[text.index("{"): text.rindex("}") + 1])
        return j if isinstance(j, dict) else None
    except ValueError:
        return None

def mk(role, text): return {"role": role, "content": text}

def tool(name, desc, *params):
    return {"type": "function", "function": {"name": name, "description": desc, "parameters": {
        "type": "object", "properties": {p: {"type": "string"} for p in params}, "required": list(params)}}}

def call_tool(registry, tc):
    """Run one tool call. A malformed call comes back as an error text, so the model can repair it. Returns (text, was_bad)."""
    try:
        args = json.loads(tc["function"]["arguments"] or "{}")
        if not isinstance(args, dict): raise ValueError("arguments must be a JSON object")
        return json.dumps(registry[tc["function"]["name"]](**args)), 0
    except Exception as e:
        return "error: " + type(e).__name__ + " " + str(e)[:80], 1

class Budget:
    """A token budget for one task. Every system of part D gets the same one. A call that starts under the cap may end over it."""
    def __init__(self, cap): self.cap, self.used, self.lock = cap, 0, threading.Lock()
    def add(self, r):
        with self.lock: self.used += r["ptok"] + r["ctok"]
    def over(self): return self.used >= self.cap

def run_agent(msgs, tools, registry, tag, budget=None, max_steps=6, model=MODEL, deadline=None, think=False, max_tokens=300):
    """The harness. Call the model, run its tool calls, feed the results back, and stop at an answer or at a limit."""
    msgs, res = list(msgs), dict(outcome="max_steps", answer="", calls=0, ptok=0, ctok=0, bad=0, tools=0, error="")
    for _ in range(max_steps):
        if budget is not None and budget.over():
            res["outcome"] = "budget"; break
        r = chat(model, msgs, tag, tools=tools, deadline=deadline, max_tokens=max_tokens, think=think)
        res["calls"] += 1; res["ptok"] += r["ptok"]; res["ctok"] += r["ctok"]
        if budget is not None: budget.add(r)
        if not r["ok"]:
            res["outcome"] = "error"; res["error"] = r["error"]; break
        if r["tool_calls"]:
            msgs.append({"role": "assistant", "content": r["content"], "tool_calls": r["tool_calls"]})
            for tc in r["tool_calls"]:
                obs, bad = call_tool(registry, tc); res["bad"] += bad; res["tools"] += 1
                msgs.append({"role": "tool", "tool_call_id": tc.get("id", ""), "content": obs})
            continue
        res["answer"] = r["content"].split("</think>")[-1].strip()
        res["outcome"] = "cut" if r["finish"] == "length" else ("answered" if res["answer"] else "empty")
        msgs.append({"role": "assistant", "content": r["content"]}); break
    res["msgs"] = msgs
    return res

def agent_outcome(res, verdict):
    """An agent run ends 'answered' or with a limit. Only an answer is graded."""
    if res["outcome"] != "answered": return res["outcome"]
    return "unparsed" if verdict is None else ("ok" if verdict else "wrong")

def check_stats():
    lo, hi = wilson(5, 10); assert 0.23 < lo < 0.24 and 0.76 < hi < 0.77 and rate_ci(5, 10) == "50 [24,76]"
    assert mcnemar([1, 1, 0, 1], [0, 0, 0, 1])[:2] == (2, 0) and abs(mcnemar([1] * 8, [0] * 8)[2] - 2 * 0.5 ** 8) < 1e-9 and mcnemar([1, 0], [1, 0])[2] == 1.0
    r = dict(ok=True, finish="stop", content="x")
    assert outcome(r, True) == "ok" and outcome(r, False) == "wrong" and outcome(r, None) == "unparsed" and outcome(dict(r, finish="length"), True) == "cut"
    assert outcome(dict(r, content=" "), True) == "empty" and outcome(dict(r, ok=False), True) == "error"
    assert agent_outcome(dict(outcome="answered"), True) == "ok" and agent_outcome(dict(outcome="budget"), True) == "budget" and maj(["a", "a", "b", "c"]) == 0.5
check_stats()


## Part A: rules, one LLM call, or an agent?

Question: on 24 shop tasks, what does an agent do that rules and one LLM call cannot do? The exam asks how agents differ from generative AI and from rule-based systems, and what an agent is made of.

Some tasks use wording that no rule expects. Some need a live value that has changed since the rules were written. Some need two tool results. Some need only arithmetic or a policy text. Every grader is deterministic. The code names the six building blocks of the agent (model, tools, memory, planning, harness, runtime) in a table.


In [ ]:
# 4. Part A: the shop world, 24 tasks, a rule system, and the grader. Pure Python. check_A() tests the world and the grader.
rng = random.Random(SEED)
ITEMS = ["lamp", "cable", "filter", "valve", "sensor", "gasket", "hinge", "bolt"]
STATUSES = ["packed", "shipped", "delivered", "returned"]
LIVE_STOCK = {i: rng.randint(6, 60) for i in ITEMS}
LIVE_ORDERS = {f"O{n}": dict(item=ITEMS[n % 8], qty=rng.randint(1, 5), price=rng.choice([12, 15, 20, 25, 40]), status=rng.choice(STATUSES))
               for n in range(1, 13)}
POLICY = ("Policy: a customer can return an item within 14 days. A refund above 100 dollars needs approval by a manager. "
          "Standard delivery takes 3 to 5 days.")
SNAP_ORDERS, SNAP_STOCK = copy.deepcopy(LIVE_ORDERS), dict(LIVE_STOCK)    # what the rule system was built from, one week ago

# TOOLS: two functions that read the live data, and their JSON schemas
def order_info(order_id): return LIVE_ORDERS.get(order_id) or {"error": "unknown order"}
def stock_level(item): return {"stock": LIVE_STOCK[item]} if item in LIVE_STOCK else {"error": "unknown item"}
A_REG = dict(order_info=order_info, stock_level=stock_level)
A_TOOLS = [tool("order_info", "Get the item, qty, unit price and status of an order.", "order_id"),
           tool("stock_level", "Get the units in stock for an item name.", "item")]

TASKS = []
def add(kind, q, answer, typ, via=None): TASKS.append(dict(id=f"{kind}{len(TASKS) + 1}", kind=kind, q=q, answer=answer, typ=typ, via=via))
st = lambda o: LIVE_ORDERS[o]["status"]
for o in ("O1", "O2"): add("exact_same", f"What is the status of order {o}?", st(o), "word", lambda o=o: order_info(o)["status"])
add("exact_same", f"How many {ITEMS[0]} units are in stock?", LIVE_STOCK[ITEMS[0]], "num", lambda: stock_level(ITEMS[0])["stock"])
for o in ("O3", "O4"):
    add("exact_changed", f"What is the status of order {o}?", st(o), "word", lambda o=o: order_info(o)["status"])
    SNAP_ORDERS[o]["status"] = STATUSES[(STATUSES.index(st(o)) + 1) % 4]    # the live status changed after the rules were written
add("exact_changed", f"How many {ITEMS[1]} units are in stock?", LIVE_STOCK[ITEMS[1]], "num", lambda: stock_level(ITEMS[1])["stock"])
SNAP_STOCK[ITEMS[1]] += 7
for o, tpl in zip(("O5", "O6", "O7", "O8"), ("Where is parcel {} right now? One word.", "Has {} shipped, been delivered, been packed or been returned? One word.",
                                              "Tell me the state of {}.", "I need the current status for {}.")):
    add("reworded", tpl.format(o), st(o), "word", lambda o=o: order_info(o)["status"])
for it, tpl in zip(ITEMS[2:6], ("Do we still have any {} left? Give the count.", "What is the count of {} on the shelf?", "Stock check for {}: how many?", "Count the {} we hold.")):
    add("reworded", tpl.format(it), LIVE_STOCK[it], "num", lambda it=it: stock_level(it)["stock"])
for o in ("O9", "O10"):
    d = LIVE_ORDERS[o]; add("multistep", f"What is the total price of order {o}?", d["qty"] * d["price"], "num", lambda o=o: order_info(o)["qty"] * order_info(o)["price"])
for o in ("O11", "O12"):
    d = LIVE_ORDERS[o]; add("multistep", f"After order {o} is picked, how many units of its item remain in stock?", LIVE_STOCK[d["item"]] - d["qty"], "num",
                            lambda o=o: stock_level(order_info(o)["item"])["stock"] - order_info(o)["qty"])
add("policy", "A client asks for a refund of 150 dollars. Which role must approve it?", "manager", "word")
add("policy", "For how many days may a client send an item back?", 14, "num")
add("policy", "What is the longest standard delivery time in days?", 5, "num")
add("arithmetic", "A 15 percent discount is applied to 80 dollars. What is the final price?", 68, "num")
add("arithmetic", "What is 12 percent tax on 250 dollars?", 30, "num")
add("arithmetic", "Split 90 dollars equally between 6 people. How much does each person get?", 15, "num")
def a_system(tools):
    return (f"You are a support agent for a shop. {POLICY} " + ("Use the tools for order and stock facts. Do not guess. " if tools else "You have no tools. ")
            + "Reply with only the final answer: one word or one number.")
VOCAB = set(STATUSES) | {"manager"}

def grade_a(task, text):
    """True, False, or None when the reply holds no number (or no known word) to grade."""
    text = text.split("</think>")[-1].strip().lower()
    if task["typ"] == "num":
        nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
        return abs(float(nums[-1]) - float(task["answer"])) < 1e-6 if nums else None
    words = set(re.findall(r"[a-z]+", text)) & VOCAB
    return words == {task["answer"]} if words else None

# The rule system: exact patterns, and answers copied from last week's data
RULES = [(re.compile(r"^What is the status of order (O\d+)\?$"), lambda m: SNAP_ORDERS[m[1]]["status"]),
         (re.compile(r"^How many (\w+) units are in stock\?$"), lambda m: SNAP_STOCK[m[1]]),
         (re.compile(r"^How many days do customers have to return an item\?$"), lambda m: 14)]
def rule_answer(q):
    for rx, f in RULES:
        m = rx.match(q)
        if m: return str(f(m))
    return None

def check_A():
    tn, tw = dict(typ="num", answer=68), dict(typ="word", answer="shipped")
    assert grade_a(tn, "The final price is $68.00.") is True and grade_a(tn, "It costs 70.") is False and grade_a(tn, "no idea") is None
    assert grade_a(tw, "Shipped.") is True and grade_a(tw, "It is delivered") is False and grade_a(tw, "shipped or delivered") is False
    assert len(TASKS) == 24 and Counter(t["kind"] for t in TASKS) == dict(exact_same=3, exact_changed=3, reworded=8, multistep=4, policy=3, arithmetic=3)
    for t in TASKS:                                   # the simulator sets the truth: the tools must return it
        if t["via"]: assert t["via"]() == t["answer"], t["id"]
        r = rule_answer(t["q"])
        if t["kind"] == "exact_same": assert r == str(t["answer"]), t["id"]
        elif t["kind"] == "exact_changed": assert r is not None and r != str(t["answer"]), t["id"]
        else: assert r is None, t["id"]
check_A()
print("world and grader checked:", dict(Counter(t["kind"] for t in TASKS)))


## Part A0: which model, which reasoning setting, for tool use?

Question: which model and setting should the agent of part A use? The exam asks for model selection (agentic.4) and for the building blocks of an agent (agentic.1).

The 14 tool tasks of part A run under three configs: the small model with reasoning off, the larger model with reasoning off, and the larger model with reasoning on. The table shows success, whether the model called a tool at all, tokens and seconds. Seconds include the wait that keeps the larger model under its request limit.


In [ ]:
# 5. Part A0: model selection for tool use. The 14 tool tasks of part A, three configs. Reasoning is a setting of the call, not of the model.
A0_CFG = dict(bolt_off=(MODEL, False, 300), omni30b_off=(UP, False, 300), omni30b_on=(UP, True, 1500))   # model, reasoning, max_tokens

def run_a0():
    dl, tasks = PART_END, [t for t in TASKS if t["via"]]
    def one(job):
        rep, cf, t = job; model, think, mt = A0_CFG[cf]; t0 = time.time()
        ag = run_agent([{"role": "system", "content": a_system(True)}, {"role": "user", "content": t["q"]}], A_TOOLS, A_REG, "A0." + cf, max_steps=6,
                       model=model, deadline=dl, think=think, max_tokens=mt)
        return fin(dict(rep=rep, conf=cf, id=t["id"], outcome=agent_outcome(ag, grade_a(t, ag["answer"])), calls=ag["calls"], ptok=ag["ptok"], ctok=ag["ctok"],
                        wall_s=time.time() - t0, tool_used=int(ag["tools"] > 0), err=ag["error"], reply=ag["answer"][:120]))
    jobs = []
    for rep in range(1, REPS_BY["A0"] + 1):
        js = [(rep, c, t) for c in A0_CFG for t in tasks]; random.Random(SEED + rep).shuffle(js); jobs += js
    df = pd.DataFrame(rows_of(jobs, pmap(one, jobs, dl), lambda j: dict(nr(rep=j[0], conf=j[1], id=j[2]["id"]), tool_used=0, err="", reply="")))
    df.to_csv(f"{OUT_DIR}/agents_A0_rows.csv", index=False); RES["A0"] = df
    t = summ(df, ["conf"], means=("tool_used",)); t["tool_used"] = (100 * t.tool_used).round().astype(int)
    show(t.rename(columns={"tool_used": "tool_%"}), "A0.1 Tool tasks of part A by config: success (percent, Wilson 95% interval), share of tasks where a tool was called at all (tool_%), cost per task", "A0_by_config")
    e = df[df.err != ""].groupby(["conf", "err"]).size().reset_index(name="count")
    show(e if len(e) else pd.DataFrame([dict(conf="all", err="none", count=0)]), "A0.2 Failed requests by config and error (after retries)", "A0_errors")
    show(pd.DataFrame([paired(df, ["rep", "id"], "conf", "omni30b_on", "omni30b_off"), paired(df, ["rep", "id"], "conf", "bolt_off", "omni30b_off"),
                       paired(df, ["rep", "id"], "conf", "omni30b_on", "bolt_off")]), "A0.3 Paired exact McNemar on the same task", "A0_paired")
    g = df.groupby("conf"); s, tu, tk = g.ok.mean(), g.tool_used.mean(), (df.ptok + df.ctok).groupby(df.conf).mean(); w = g.wall_s.mean()
    say(f"Reading from the numbers: best success is {s.idxmax()} at {100 * s.max():.0f}%. Nano-omni 30B with reasoning off solves {100 * s['omni30b_off']:.0f}% and calls a tool in {100 * tu['omni30b_off']:.0f}% of tasks; "
        f"with reasoning on it solves {100 * s['omni30b_on']:.0f}% and calls a tool in {100 * tu['omni30b_on']:.0f}%, at {tk['omni30b_on']:.0f} tokens and {w['omni30b_on']:.1f} s per task. "
        f"Bolt with reasoning off solves {100 * s['bolt_off']:.0f}% at {tk['bolt_off']:.0f} tokens and {w['bolt_off']:.1f} s.")
    samples(df)

part("A0", run_a0)


In [ ]:
# 6. Part A: rules, one LLM call, and an agent on the same 24 tasks, twice. The agent code is below, with its six building blocks named.
BLOCKS = pd.DataFrame([
    ("model", "MODEL and chat()", "The LLM. It reads text and picks the next step."),
    ("tools", "A_TOOLS, A_REG", "Functions the model can ask for. The code runs them, never the model."),
    ("memory", "msgs inside run_agent()", "The chat so far. It holds the task, each tool call and each result."),
    ("planning", "the loop with max_steps", "The model chooses a tool, reads the result, and chooses again (ReAct)."),
    ("harness", "run_agent()", "Scaffolding around the model: it runs tool calls, feeds results back, and stops at a limit."),
    ("runtime", "pmap, SEM, CALL_CAP, deadline", "Where the agent runs, with limits on time, calls and requests. It is not a sandbox here."),
], columns=["block", "in this code", "what it does"])

def run_a():
    dl, rng = PART_END, random.Random(SEED)
    def one(job):
        rep, cond, t = job; t0 = time.time(); base = dict(rep=rep, cond=cond, kind=t["kind"], id=t["id"])
        if cond == "rules":
            a = rule_answer(t["q"]); v = None if a is None else grade_a(t, a)
            return fin(dict(base, outcome="no_rule" if a is None else ("ok" if v else "wrong"), calls=0, ptok=0, ctok=0, wall_s=0.0, reply=a or "", err=""))
        msgs = [{"role": "system", "content": a_system(cond == "agent")}, {"role": "user", "content": t["q"]}]
        if cond == "one_call":
            r = chat(MODEL, msgs, "A.one_call", max_tokens=60, deadline=dl)
            return fin(dict(base, outcome=outcome(r, grade_a(t, r["content"])), calls=1, ptok=r["ptok"], ctok=r["ctok"], wall_s=time.time() - t0, reply=r["content"][:120], err=r["error"]))
        ag = run_agent(msgs, A_TOOLS, A_REG, "A.agent", max_steps=6, deadline=dl)
        return fin(dict(base, outcome=agent_outcome(ag, grade_a(t, ag["answer"])), calls=ag["calls"], ptok=ag["ptok"], ctok=ag["ctok"],
                        wall_s=time.time() - t0, reply=ag["answer"][:120], err=ag["error"], bad_tool_calls=ag["bad"]))
    jobs = [j for rep in range(1, REPS_BY["A"] + 1) for j in random.Random(SEED + rep).sample([(rep, c, t) for c in ("rules", "one_call", "agent") for t in TASKS], 72)]
    df = pd.DataFrame(rows_of(jobs, pmap(one, jobs, dl), lambda j: nr(rep=j[0], cond=j[1], kind=j[2]["kind"], id=j[2]["id"])))
    df.to_csv(f"{OUT_DIR}/agents_A_rows.csv", index=False); RES["A"] = df
    show(BLOCKS, "Building blocks of the agent in this notebook (source: ai-agents.txt)", "A_blocks")
    show(summ(df, ["cond"]), "A1. Same 24 tasks, success in percent with Wilson 95% interval, two repeats (r1, r2), mean cost per task", "A_by_system")
    print(f"baseline: always the most common answer would score {100 * maj([t['answer'] for t in TASKS]):.0f}%")
    show(summ(df, ["kind", "cond"]).drop(columns=["calls", "tok", "wall_s"]), "A2. Success by task kind", "A_by_kind")
    show(pd.DataFrame([paired(df, ["rep", "id"], "cond", "agent", "one_call"), paired(df, ["rep", "id"], "cond", "agent", "rules"),
                       paired(df, ["rep", "id"], "cond", "one_call", "rules")]), "A3. Paired exact McNemar tests (same task and repeat). Repeats of one task are not independent: read p as a guide.", "A_paired")
    g = df.groupby("cond"); s = g.ok.mean(); c = g.calls.mean(); k = (df.ptok + df.ctok).groupby(df.cond).mean()
    say(f"Reading from the numbers: the agent solves {100 * s['agent']:.0f}% of tasks, one call {100 * s['one_call']:.0f}%, rules {100 * s['rules']:.0f}%. "
          f"The agent needs {c['agent']:.1f} calls and {k['agent']:.0f} tokens per task, one call needs 1 call and {k['one_call']:.0f} tokens, rules need 0.")
    samples(df)
    ex = df[(df.cond == "rules") & (df.kind == "exact_changed")]
    print(f"Rules on tasks with a changed live value: {int(ex.ok.sum())} of {len(ex)} right. Their answers come from last week's data.")

part("A", run_a)


## Part B: state and context

Question: if a conversation is long, what do you put in the context window? The exam asks about stateless and stateful interactions and context engineering.

Ten conversations have 6 turns. Turns 1 to 5 each state one fact. Turn 6 is the late turn. It asks for two early facts and one recent fact. Five strategies build the context: stateless (the question only), the full history, a sliding window (turns 4 to 6), a summary, and a structured state object. The noisy version adds 30 irrelevant sentences to each turn. Some of them are decoys: other people's badge numbers, rooms and dates.


In [ ]:
# 7. Part B: ten conversations of 6 turns, a clean and a noisy version, five ways to build the context. Pure Python.
rngb = random.Random(SEED + 1)
SLOTS = ["badge", "room", "supplier", "budget", "deadline"]            # user turn k (1 to 5) states slot k. Turn 6 asks.
SUP = ["Norvik", "Alder", "Brightway", "Carrow", "Delmar", "Evertt", "Fenwick", "Garrick", "Holloway", "Ironwood"]
MON, WHO = ["March", "April", "May", "June", "July", "August"], ["Dana", "Marta", "Priya", "Tomas", "Lena", "Omar", "Ines", "Yusuf"]
FACT = dict(badge="My badge number is {}.", room="My meeting room is {}.", supplier="My preferred supplier is {}.",
            budget="My budget is {} dollars.", deadline="My deadline is {}.")
DECOY = dict(badge="{w}'s badge number is {v}.", room="{w} booked room {v} for a review.", supplier="{w} says the usual supplier is {v}.",
             budget="{w} has a budget of {v} dollars.", deadline="{w}'s deadline is {v}.")
FILL = ["The cafeteria changed its menu this week.", "The elevator in the north wing is slow.", "Parking on level two is full by nine.",
        "A printer on the third floor needs paper.", "The quarterly newsletter is out on Friday.", "Someone left an umbrella in the lobby.",
        "The fire drill is next Tuesday.", "The new intern starts on Monday."]
OPEN = ["Thanks for the help today.", "One more thing for my notes.", "Let me add a detail.", "Here is another detail.", "Last detail for now."]
Q6 = "Reply as JSON with the keys badge, room and deadline. What is my badge number, my meeting room, and my deadline?"
SYS_B = "You are a helpful assistant. Use only the conversation or memory you are given. Reply with a JSON object only. If you do not know a value, use \"unknown\"."
N_NOISE = 30                                                            # noise sentences per turn in the noisy version

def draw(rng):
    return dict(badge=str(rng.randint(1000, 9999)), room=f"{rng.choice('ABCDEF')}-{rng.randint(10, 99)}", supplier=rng.choice(SUP),
                budget=str(rng.randint(30, 95) * 100), deadline=f"{rng.choice(MON)} {rng.randint(2, 28)}")

def make_conv(i):
    truth = draw(rngb)
    while truth["badge"] in {c["truth"]["badge"] for c in CONVS}: truth = draw(rngb)
    clean, noisy = [], []
    for k, slot in enumerate(SLOTS):
        fact = f"{OPEN[k]} {FACT[slot].format(truth[slot])}"
        clean.append(fact)
        sents = []
        for _ in range(N_NOISE):                                       # decoys have the same shape as the truth but belong to other people
            if rngb.random() < 0.6:
                s = rngb.choice(SLOTS); v = draw(rngb)[s]
                while v in truth.values(): v = draw(rngb)[s]
                sents.append(DECOY[s].format(w=rngb.choice(WHO), v=v))
            else:
                sents.append(rngb.choice(FILL))
        sents.insert(rngb.randint(0, len(sents)), FACT[slot].format(truth[slot]))
        noisy.append(OPEN[k] + " " + " ".join(sents))
    return dict(id=i, truth=truth, turns=dict(clean=clean, noisy=noisy))
CONVS = []
for i in range(10): CONVS.append(make_conv(i))

def transcript(turns): return "\n".join(f"User: {u}\nAssistant: Noted." for u in turns)
def norm(x): return re.sub(r"[\s$,]", "", str(x)).lower()
def grade_b(conv, text):
    """Returns (early hits of 2, recent hits of 1), or None when the reply is not a JSON object."""
    j = jparse(text)
    if j is None: return None
    t = conv["truth"]; hit = lambda s: int(norm(j.get(s, "")) == norm(t[s]))
    return hit("badge") + hit("room"), hit("deadline")

def check_B():
    for c in CONVS:
        for var in ("clean", "noisy"):
            text = " ".join(c["turns"][var])
            for k, slot in enumerate(SLOTS):               # the truth is said once, in its own turn, and no decoy repeats it
                v = c["truth"][slot]
                assert len(re.findall(r"(?<![\w-])" + re.escape(v) + r"(?![\w-])", text)) == 1, (c["id"], var, slot)
                assert v in c["turns"][var][k]
    c = CONVS[0]; t = c["truth"]
    good = json.dumps(dict(badge=t["badge"], room=t["room"], deadline=t["deadline"]))
    assert grade_b(c, good) == (2, 1) and grade_b(c, "```json\n" + good + "\n```") == (2, 1)
    assert grade_b(c, json.dumps(dict(badge="0000", room=t["room"], deadline="x"))) == (1, 0) and grade_b(c, "unknown") is None
check_B()
print("conversations checked:", len(CONVS), "| tokens of a noisy conversation, about", len(" ".join(CONVS[0]["turns"]["noisy"])) // 4)


In [ ]:
# 8. Part B: five strategies, clean and noisy, twice. Turn 6 asks for two early facts (turns 1 and 2) and one recent fact (turn 5).
STRATS = ["stateless", "full_history", "sliding_window", "summary", "state_object"]

def run_b():
    dl = PART_END
    def prep(strat, turns, tag):
        """The strategies that rewrite the history: one extra model call each. Returns (memory text, call result)."""
        if strat == "summary":
            r = chat(MODEL, [{"role": "system", "content": "Summarize every fact the user states about themselves in at most 60 words."},
                             {"role": "user", "content": transcript(turns)}], tag, max_tokens=150, deadline=dl)
            return r["content"].strip(), r
        r = chat(MODEL, [{"role": "system", "content": "Extract the user's own facts as one JSON object with the keys badge, room, supplier, budget, deadline. "
                          "Use null for a fact not stated. Reply with JSON only."}, {"role": "user", "content": transcript(turns)}], tag, max_tokens=150, deadline=dl)
        return json.dumps(jparse(r["content"]) or {}), r

    def one(job):
        rep, var, strat, c = job; t0 = time.time(); turns = c["turns"][var]
        hist = [m for u in turns for m in ({"role": "user", "content": u}, {"role": "assistant", "content": "Noted."})]
        q, sysm, mem, pr = {"role": "user", "content": Q6}, SYS_B, "", dict(ptok=0, ctok=0, calls=0)
        if strat in ("summary", "state_object"):
            mem, pr = prep(strat, turns, "B.prep"); pr = dict(ptok=pr["ptok"], ctok=pr["ctok"], calls=1)
            sysm += ("\nSummary of the conversation so far: " if strat == "summary" else "\nState object: ") + mem
        msgs = {"stateless": [], "full_history": hist, "sliding_window": hist[-4:]}.get(strat, [])   # window = turns 4 and 5, then the question
        r = chat(MODEL, [{"role": "system", "content": sysm}] + msgs + [q], "B.answer", max_tokens=120, deadline=dl)
        g = grade_b(c, r["content"]); e, l = g or (0, 0)
        return fin(dict(rep=rep, var=var, strat=strat, conv=c["id"], cond=f"{strat}|{var}", outcome=outcome(r, None if g is None else g == (2, 1)),
                        hits=e + l, items=3, e_hits=e, e_items=2, l_hits=l, l_items=1, calls=1 + pr["calls"], ans_ptok=r["ptok"],
                        ptok=r["ptok"] + pr["ptok"], ctok=r["ctok"] + pr["ctok"], wall_s=time.time() - t0, reply=r["content"][:120]))
    jobs = [j for rep in range(1, REPS_BY["B"] + 1) for j in random.Random(SEED + rep).sample(
        [(rep, v, s, c) for v in ("clean", "noisy") for s in STRATS for c in CONVS if not (v == "noisy" and s == "stateless")], 90)]
    # stateless has no history, so the noisy version would be the same request. It is not run twice. Its row is copied below.
    df = pd.DataFrame(rows_of(jobs, pmap(one, jobs, dl), lambda j: dict(nr(rep=j[0], var=j[1], strat=j[2], conv=j[3]["id"], cond=f"{j[2]}|{j[1]}"), hits=0, items=3, e_hits=0, e_items=2, l_hits=0, l_items=1, ans_ptok=0)))
    dn = df[(df.strat == "stateless")].copy(); dn["var"], dn["cond"] = "noisy", "stateless|noisy"; df = pd.concat([df, dn], ignore_index=True)
    df.to_csv(f"{OUT_DIR}/agents_B_rows.csv", index=False); RES["B"] = df
    specs = (("all", "hits", "items"), ("early", "e_hits", "e_items"), ("late", "l_hits", "l_items"))
    show(summ(df, ["var", "strat"], specs, means=("ans_ptok",)).drop(columns=["calls", "wall_s"]), "B1. Accuracy per fact (percent, Wilson 95% interval), tokens per conversation (tok), prompt tokens of the answer call (ans_ptok)", "B_by_strategy")
    ch = {s: maj([c["truth"][s] for c in CONVS]) for s in ("badge", "room", "deadline")}
    print(f"baseline: always the most common answer scores {100 * ch['badge']:.0f}% on badge, {100 * ch['room']:.0f}% on room, {100 * ch['deadline']:.0f}% on deadline. Stateless is the empirical floor.")
    g = df.groupby(["strat", "var"], observed=True); k, n = g.e_hits.sum().unstack(), g.e_items.sum().unstack()
    rot = pd.DataFrame({"strat": STRATS, "clean": [rate_ci(k.loc[s, "clean"], n.loc[s, "clean"]) for s in STRATS], "noisy": [rate_ci(k.loc[s, "noisy"], n.loc[s, "noisy"]) for s in STRATS],
                        "rot_points": [round(100 * (k.loc[s, "clean"] / n.loc[s, "clean"] - k.loc[s, "noisy"] / n.loc[s, "noisy"]), 1) for s in STRATS]})
    show(rot, "B2. Context rot: early-fact accuracy (percent, Wilson 95% interval), clean against noisy. rot_points = clean minus noisy.", "B_rot")
    show(pd.DataFrame([paired(df, ["rep", "conv"], "cond", "state_object|noisy", "full_history|noisy"), paired(df, ["rep", "conv"], "cond", "full_history|clean", "full_history|noisy"),
                       paired(df, ["rep", "conv"], "cond", "full_history|clean", "sliding_window|clean")]), "B3. Paired exact McNemar (all three facts right in one reply)", "B_paired")
    samples(df)
    f, s = df[df.cond == "full_history|noisy"], df[df.cond == "state_object|noisy"]
    say(f"Reading from the numbers: on noisy conversations the full history costs {f.ans_ptok.mean():.0f} prompt tokens and gets {100 * f.e_hits.sum() / f.e_items.sum():.0f}% of early facts; "
          f"the state object costs {s.ans_ptok.mean():.0f} prompt tokens in the answer call ({s.ptok.mean():.0f} with the extraction call) and gets {100 * s.e_hits.sum() / s.e_items.sum():.0f}%.")

part("B", run_b)


## Part C: short-term and long-term memory

Question: which memory does an agent need, and what breaks? The exam asks for short-term and long-term memory and when each is right.

Five users talk to the agent in three sessions. One fact changes between sessions 1 and 2. One fact is told only in session 3. One fact is told once in session 1. Six stores are compared: none (the current session only), the whole log, BM25 keyword retrieval, and a key-value profile that is overwritten. Two stores have no user scope: BM25 and the profile, shared by all users. A stale answer is the value before the change. A leak is a value that belongs to another user.


In [ ]:
# 9. Part C: five users, three sessions each, a fact that changes, six memory stores, BM25 in plain Python. Pure Python.
rngc = random.Random(SEED + 2)
CITY = ["Oslo", "Lima", "Turin", "Graz", "Porto", "Leeds", "Bergen", "Malaga", "Lyon", "Gdansk", "Cork", "Ghent"]
OSES = ["Linux", "Windows", "macOS", "ChromeOS", "FreeBSD", "Solaris"]
FILL_C = ["The coffee machine is broken again.", "I walked to work today.", "The meeting ran long.", "It rained all week.", "I need a new chair.", "Lunch was late."]
cities, oses = rngc.sample(CITY, 10), rngc.sample(OSES, 5)
USERS = []
for k in range(5):
    u = dict(id=f"u{k + 1}", old=cities[2 * k], new=cities[2 * k + 1], os=oses[k], ticket=f"TK-{1000 + 997 * k + rngc.randint(0, 99)}")
    u["sessions"] = {1: [f"I live in {u['old']}.", f"My operating system is {u['os']}.", rngc.choice(FILL_C)],
                     2: [f"I moved, so I live in {u['new']} now.", rngc.choice(FILL_C)], 3: [f"My ticket number today is {u['ticket']}.", rngc.choice(FILL_C)]}
    USERS.append(u)
DOCS = [dict(user=u["id"], s=s, text=t) for u in USERS for s in (1, 2, 3) for t in u["sessions"][s]]    # the long-term text store
QS = dict(same=("What is my ticket number today?", "ticket"), changed=("In which city do I live now?", "city"), stable=("Which operating system do I use?", "os"))
SLOT_VALUES = lambda u, slot: dict(ticket=[u["ticket"]], city=[u["old"], u["new"]], os=[u["os"]])[slot]
SYS_C = ("You answer questions about the user. Use only the memory below. Newer sessions replace older ones. "
         "If the memory has no answer, reply unknown. Reply with the answer only.\nMemory:\n")

def bm25_top(query, docs, k=3, k1=1.5, b=0.75):
    """Keyword retrieval: BM25 score of each doc for the query. Ties keep the order of the store."""
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    D = [tok(d["text"]) for d in docs]; N = len(D); avg = sum(map(len, D)) / N
    df = Counter(w for d in D for w in set(d)); q = tok(query)
    def score(d):
        tf = Counter(d)
        return sum(math.log(1 + (N - df[w] + .5) / (df[w] + .5)) * tf[w] * (k1 + 1) / (tf[w] + k1 * (1 - b + b * len(d) / avg)) for w in q if w in tf)
    return [docs[i] for i in sorted(range(N), key=lambda i: -score(D[i]))[:k]]

def context(store, u, q, prof, shared):
    mine = [d for d in DOCS if d["user"] == u["id"]]
    show_ = lambda ds: "\n".join(f"[S{d['s']}] {d['text']}" for d in ds)
    return {"none": lambda: "\n".join(f"[S3] {s}" for s in u["sessions"][3]),         # short-term memory: this session only
            "dump": lambda: "\n".join(f"[S{k}] {s}" for k in (1, 2, 3) for s in u["sessions"][k]),   # long-term: the whole log
            "bm25": lambda: show_(bm25_top(q, mine)), "bm25_shared": lambda: show_(bm25_top(q, DOCS)),               # long-term: retrieval
            "kv": lambda: json.dumps(prof[u["id"]]), "kv_shared": lambda: json.dumps(shared)}[store]()               # long-term: profile

def grade_mem(ans, cur, old, others):
    """Name the answer: ok, stale (the value before the change), leak (a value of another user), or wrong."""
    m = {v for v in [cur, old, *others] if v and re.search(r"(?<![\w-])" + re.escape(v.lower()) + r"(?![\w-])", ans.lower())}
    if m == {cur}: return "ok"
    if old and old in m and cur not in m and m == {old}: return "stale"
    if m & set(others) and cur not in m: return "leak"
    return "wrong"

def check_C():
    docs = [dict(text="I live in Oslo."), dict(text="My ticket is 5."), dict(text="The coffee is cold.")]
    assert bm25_top("In which city do I live?", docs, 1)[0] is docs[0] and bm25_top("What is my ticket?", docs, 1)[0] is docs[1]
    assert grade_mem("Lima", "Lima", "Oslo", ["Graz"]) == "ok" and grade_mem("You live in Oslo.", "Lima", "Oslo", ["Graz"]) == "stale"
    assert grade_mem("Graz", "Lima", "Oslo", ["Graz"]) == "leak" and grade_mem("Oslo or Lima", "Lima", "Oslo", []) == "wrong" and grade_mem("unknown", "Lima", "Oslo", []) == "wrong"
    for u in USERS:                                   # the simulator sets the truth: the old city is in session 1, the new one only in session 2
        txt = {s: " ".join(u["sessions"][s]) for s in (1, 2, 3)}
        assert u["old"] in txt[1] and u["new"] in txt[2] and u["old"] != u["new"] and u["ticket"] in txt[3] and u["os"] in txt[1]
        assert u["new"] not in txt[1] and u["old"] not in txt[2] and u["ticket"] not in txt[1] + txt[2]
    assert len({u["ticket"] for u in USERS}) == 5 == len({u["os"] for u in USERS}) and len({u[k] for u in USERS for k in ("old", "new")}) == 10
check_C()
print("memory world and graders checked:", len(DOCS), "stored sentences for", len(USERS), "users")


In [ ]:
# 10. Part C: short-term against long-term memory, four stores with a user scope and two without. Twice.
SCOPED, SHARED = ["none", "dump", "bm25", "kv"], ["bm25_shared", "kv_shared"]

def run_c():
    dl = PART_END
    def extract(job):                           # the key-value store: one model call per finished session writes facts into a profile
        rep, u, s = job
        r = chat(MODEL, [{"role": "system", "content": "Extract the facts the user states about themselves as one JSON object with the keys city, os, ticket. "
                          "Leave out a key that is not stated. Reply with JSON only."}, {"role": "user", "content": "\n".join(u["sessions"][s])}], "C.extract", max_tokens=80, deadline=dl)
        j = jparse(r["content"]) or {}
        return dict(u=u["id"], s=s, facts={k: str(v) for k, v in j.items() if k in ("city", "os", "ticket") and v}, ptok=r["ptok"], ctok=r["ctok"])
    def one(job):
        rep, store, u, kind, prof, shared = job; t0 = time.time(); q, slot = QS[kind]
        cur = dict(ticket=u["ticket"], city=u["new"], os=u["os"])[slot]; old = u["old"] if slot == "city" else None
        others = [v for w in USERS if w is not u for v in SLOT_VALUES(w, slot)]
        r = chat(MODEL, [{"role": "system", "content": SYS_C + context(store, u, q, prof, shared)}, {"role": "user", "content": q}], "C.answer", max_tokens=40, deadline=dl)
        cls = grade_mem(r["content"], cur, old, others); oc = outcome(r, cls == "ok")
        return fin(dict(rep=rep, store=store, user=u["id"], qkind=kind, outcome=cls if oc == "wrong" and cls in ("stale", "leak") else oc, calls=1, ptok=r["ptok"], ctok=r["ctok"],
                        wall_s=time.time() - t0, reply=r["content"][:120]))
    rows, internals, ext_tok = [], [], {}
    for rep in range(1, REPS_BY["C"] + 1):
        ex = [r for r in pmap(extract, [(rep, u, s) for u in USERS for s in (1, 2, 3)], dl) if r]
        prof, shared = {u["id"]: {} for u in USERS}, {}
        for e in sorted(ex, key=lambda e: (e["u"], e["s"])):     # a profile is overwritten in session order. The shared one is overwritten by every user.
            prof[e["u"]].update(e["facts"]); shared.update(e["facts"])
        truth = {u["id"]: dict(city=u["new"], os=u["os"], ticket=u["ticket"]) for u in USERS}
        internals.append(dict(measure=f"kv profile holds the true fact (repeat {rep})", scoped=rate_ci(sum(prof[u].get(k) == truth[u][k] for u in truth for k in truth[u]), 15),
                              shared=rate_ci(sum(shared.get(k) == truth[u][k] for u in truth for k in truth[u]), 15)))
        ext_tok[rep] = sum(e["ptok"] + e["ctok"] for e in ex)
        jobs = [(rep, s, u, k, prof, shared) for s in SCOPED + SHARED for u in USERS for k in QS if not (s in SHARED and k == "same")]
        jobs = random.Random(SEED + rep).sample(jobs, len(jobs))
        new = rows_of(jobs, pmap(one, jobs, dl), lambda j: nr(rep=j[0], store=j[1], user=j[2]["id"], qkind=j[3]))
        for s in ("kv", "kv_shared"):                            # the extraction tokens belong to the store that needs them
            mine = [r for r in new if r["store"] == s]
            for r in mine: r["ptok"] += ext_tok[rep] / len(mine)
        rows += new
    df = pd.DataFrame(rows); df.to_csv(f"{OUT_DIR}/agents_C_rows.csv", index=False); RES["C"] = df
    show(summ(df, ["store", "qkind"]).drop(columns=["calls", "wall_s"]), "C1. Answer quality by store and question (same = said in this session, changed = city, stable = said once). tok includes the extraction calls of kv.", "C_by_store")
    print(f"baseline: always the most common answer scores {100 * maj([u['ticket'] for u in USERS]):.0f}% on every question (five users, five different answers).")
    cross = df[df.qkind != "same"]; rows2 = []
    for s, d in cross.groupby("store", sort=False):
        r = dict(store=s, n=len(d))
        for oc in ("ok", "stale", "leak", "wrong"):
            k = int((d.outcome == oc).sum()); lo, hi = wilson(k, len(d)); r[oc] = f"{100 * k / len(d):.0f} [{100 * lo:.0f},{100 * hi:.0f}]"
        rows2.append(r)
    show(pd.DataFrame(rows2), "C2. Questions that need an earlier session: share of ok, stale, leak (another user's value) and wrong answers", "C_stale_leak")
    pairs = [(u, k) for u in USERS for k in ("changed", "stable")]
    retr = lambda shared_: [bm25_top(QS[k][0], [d for d in DOCS if shared_ or d["user"] == u["id"]]) for u, k in pairs]
    hit = lambda shared_: (lambda v: rate_ci(sum(v), len(v)))([any(SLOT_VALUES(u, QS[k][1])[-1] in d["text"] for d in t) for t, (u, k) in zip(retr(shared_), pairs)])
    foreign = lambda shared_: (lambda v: rate_ci(sum(v), len(v)))([d["user"] != u["id"] for t, (u, k) in zip(retr(shared_), pairs) for d in t])
    internals += [dict(measure="bm25 top 3 holds the sentence with the true value", scoped=hit(False), shared=hit(True)),
                  dict(measure="bm25 top 3 sentences that belong to another user", scoped=foreign(False), shared=foreign(True))]
    show(pd.DataFrame(internals), "C3. Store internals, no model call (percent, Wilson 95% interval).", "C_internals")
    show(pd.DataFrame([paired(cross, ["rep", "user", "qkind"], "store", "kv", "bm25"), paired(cross, ["rep", "user", "qkind"], "store", "dump", "bm25"),
                       paired(df[df.qkind == "same"], ["rep", "user", "qkind"], "store", "none", "dump")]), "C4. Paired exact McNemar. The last row is the same-session question: no store needed.", "C_paired")
    samples(df)
    g = cross.groupby("store"); s = g.ok.mean(); sd = (cross.outcome == "stale").groupby(cross.store).mean(); lk = (cross.outcome == "leak").groupby(cross.store).mean()
    say(f"Reading from the numbers: no store gets {100 * s['none']:.0f}% of the cross-session questions; the whole log gets {100 * s['dump']:.0f}%, BM25 {100 * s['bm25']:.0f}%, the profile {100 * s['kv']:.0f}%. "
          f"Stale answers: BM25 {100 * sd['bm25']:.0f}%, profile {100 * sd['kv']:.0f}%. Leaks without a user scope: BM25 {100 * lk['bm25_shared']:.0f}%, profile {100 * lk['kv_shared']:.0f}%.")

part("C", run_c)


## Part E: improving an agent

Question: how do you make an agent better, and how do you know the gain is real? The exam asks for feedback, prompt refinement, workflow refinement and model selection.

A router reads a support ticket and replies with a queue and a priority. A hidden policy sets the right answer (for example a charge you do not recognize goes to security). The loop starts with the small model and a vague prompt. It makes one change at a time: a check step in the workflow, a prompt that a larger model writes from the dev errors, and the larger model of part A0. A change stays only if the dev score rises. The held-out tickets only measure.


In [ ]:
# 11. Part E: a ticket router with a hidden policy, a dev set and a held-out set, a strict grader. Pure Python. check_E() tests both.
rnge = random.Random(SEED + 4)
QUEUES, PRIOS = ["billing", "security", "shipping", "general"], ["high", "normal"]
P0 = "You route support tickets. Reply as JSON with the keys queue and priority."

def policy(topic, amount=0, days=0, unrecognized=False):
    """The hidden rule the router must find. Only the grader sees it."""
    if unrecognized or topic == "security": return "security", "high"
    if topic == "billing": return "billing", "high" if amount >= 500 else "normal"
    if topic == "shipping": return "shipping", "high" if days >= 7 else "normal"
    return "general", "normal"

def mk_ticket(kind, i):
    ch = rnge.choice
    if kind in ("bill_low", "bill_high"):
        a = rnge.randint(40, 450) if kind == "bill_low" else rnge.randint(520, 2400)
        text = ch([f"I was charged {a} dollars twice for one order.", f"Please refund the {a} dollars I paid for a plan I cancelled.", f"My invoice shows {a} dollars but the quote said less."])
        return dict(id=f"t{i}", kind=kind, text=text, queue="billing", priority="normal" if kind == "bill_low" else "high", attrs=dict(topic="billing", amount=a))
    if kind == "fraud":
        a = rnge.randint(50, 900)
        return dict(id=f"t{i}", kind=kind, text=f"There is a charge of {a} dollars on my card that I do not recognize.", queue="security", priority="high", attrs=dict(topic="billing", amount=a, unrecognized=True))
    if kind in ("ship_low", "ship_high"):
        d = rnge.randint(1, 5) if kind == "ship_low" else rnge.randint(8, 20)
        text = ch([f"My parcel is {d} days late.", f"The package has not arrived after {d} days.", f"Tracking shows no update for {d} days."])
        return dict(id=f"t{i}", kind=kind, text=text, queue="shipping", priority="normal" if kind == "ship_low" else "high", attrs=dict(topic="shipping", days=d))
    if kind == "hack":
        return dict(id=f"t{i}", kind=kind, text=ch(["Someone logged into my account from another country.", "I think my account was hacked."]), queue="security", priority="high", attrs=dict(topic="security"))
    return dict(id=f"t{i}", kind=kind, text=ch(["How do I change the email address on my profile?", "Where can I find the opening hours of your store?"]), queue="general", priority="normal", attrs=dict(topic="general"))

KINDS = ["bill_low", "bill_high", "fraud", "ship_low", "ship_high", "hack", "general"]
dev_k = KINDS + ["bill_high", "fraud", "ship_high"]; test_k = KINDS * 2 + ["bill_low", "ship_low", "fraud", "bill_high", "ship_high", "hack"]
rnge.shuffle(dev_k); rnge.shuffle(test_k)
DEV = [mk_ticket(k, i) for i, k in enumerate(dev_k)]; TEST = [mk_ticket(k, 100 + i) for i, k in enumerate(test_k)]

def grade_e(t, text):
    """Returns (verdict, category). Strict: the whole reply must be one JSON object with exactly the two keys and allowed values."""
    try: j = json.loads(text.strip())
    except ValueError: return None, "invalid_json"
    if not isinstance(j, dict) or set(j) != {"queue", "priority"} or j["queue"] not in QUEUES or j["priority"] not in PRIOS: return None, "bad_value"
    q, p = j["queue"] == t["queue"], j["priority"] == t["priority"]
    return (True, "ok") if q and p else (False, "wrong_both" if not q and not p else "wrong_queue" if not q else "wrong_priority")

def check_E():
    assert policy("billing", 500) == ("billing", "high") and policy("billing", 499) == ("billing", "normal") and policy("shipping", days=7) == ("shipping", "high")
    assert policy("shipping", days=6) == ("shipping", "normal") and policy("billing", 100, unrecognized=True) == ("security", "high") and policy("general") == ("general", "normal")
    for t in DEV + TEST: assert (t["queue"], t["priority"]) == policy(**t["attrs"]), t["id"]          # the label of each ticket follows the policy
    t = dict(queue="billing", priority="high")
    assert grade_e(t, '{"queue": "billing", "priority": "high"}') == (True, "ok") and grade_e(t, '{"queue": "billing", "priority": "normal"}') == (False, "wrong_priority")
    assert grade_e(t, '```json\n{"queue": "billing", "priority": "high"}\n```') == (None, "invalid_json") and grade_e(t, '{"queue": "billing", "priority": "urgent"}') == (None, "bad_value")
    assert grade_e(t, '{"queue": "general", "priority": "normal"}') == (False, "wrong_both") and grade_e(t, '{"queue": "general", "priority": "high"}') == (False, "wrong_queue")
    assert len(DEV) == 10 and len(TEST) == 20
check_E()
print("tickets checked: dev", Counter(t["kind"] for t in DEV).most_common(3), "... held-out", len(TEST))


In [ ]:
# 12. Part E: the improvement loop, one change at a time, twice. A change is kept only if it raises the dev score. The held-out set only measures.
def run_e():
    dl, items = PART_END, [("dev", t) for t in DEV] + [("test", t) for t in TEST]
    def route(cfg, t):
        """One ticket. The check step (workflow refinement) is code: if the reply is invalid, ask once more with the allowed values."""
        msgs, tot = [mk("system", cfg["prompt"]), mk("user", t["text"])], dict(calls=1, ptok=0, ctok=0)
        r = chat(cfg["model"], msgs, "E." + cfg["name"], max_tokens=120, deadline=dl); v, cat = grade_e(t, r["content"]) if r["ok"] else (None, "error")
        tot["ptok"], tot["ctok"] = r["ptok"], r["ctok"]
        if cfg["check"] and cat in ("invalid_json", "bad_value"):
            r2 = chat(cfg["model"], msgs + [mk("assistant", r["content"]), mk("user", f"Your reply was not valid ({cat}). Reply with raw JSON only, with the keys queue "
                                                                                       "(billing, security, shipping or general) and priority (high or normal).")], "E." + cfg["name"], max_tokens=120, deadline=dl)
            tot["calls"] += 1; tot["ptok"] += r2["ptok"]; tot["ctok"] += r2["ctok"]; r = r2; v, cat = grade_e(t, r["content"]) if r["ok"] else (None, "error")
        return dict(outcome=outcome(r, v), cat=cat, reply=r["content"][:120], **tot)
    def evaluate(cfg, rep):
        jobs = [(s, t) for s, t in items]
        def one(j):
            t0 = time.time(); r = route(cfg, j[1])
            return fin(dict(rep=rep, cfg=cfg["name"], split=j[0], id=j[1]["id"], wall_s=time.time() - t0, **r))
        return rows_of(jobs, pmap(one, jobs, dl), lambda j: dict(nr(rep=rep, cfg=cfg["name"], split=j[0], id=j[1]["id"]), cat="not_reached", reply=""))
    acc = lambda rows, s: np.mean([r["ok"] for r in rows if r["split"] == s])
    def analyse(rows):
        by = {t["id"]: t for _, t in items}; errs = [r for r in rows if r["split"] == "dev" and not r["ok"]]
        if not errs or time.time() >= dl: return None
        lines = "\n".join(f"Ticket: {by[e['id']]['text']}\nModel reply: {e['reply']}\nCorrect: queue={by[e['id']]['queue']}, priority={by[e['id']]['priority']}" for e in errs[:10])
        r = chat(BIG, [mk("system", "You improve prompts. You get support tickets that a router got wrong, with the model reply and the correct answer. Write a better system prompt "
                          "(at most 120 words) for the router. State the allowed values, the output format, and the rules you can infer from the errors. Reply with the prompt text only."),
                       mk("user", lines)], "E.analysis", max_tokens=300, deadline=dl)
        return r["content"].strip()[:1200] if r["ok"] and r["content"].strip() else None
    rows, log = [], []
    for rep in range(1, REPS_BY["E"] + 1):
        kept = dict(name="c0_baseline", prompt=P0, model=MODEL, check=False)
        cur = evaluate(kept, rep); rows += cur; kept_rows, kept_dev = cur, acc(cur, "dev"); log.append(dict(rep=rep, cfg=kept["name"], dev=round(float(kept_dev), 2), kept="start"))
        for name, change in (("c1_check_step", lambda k: dict(check=True)), ("c2_prompt", lambda k: dict(prompt=analyse(kept_rows) or k["prompt"])), ("c3_model", lambda k: dict(model=UP))):
            cfg = {**kept, **change(kept), "name": name}; cur = evaluate(cfg, rep); rows += cur; d = acc(cur, "dev"); keep = d > kept_dev
            log.append(dict(rep=rep, cfg=name, dev=round(float(d), 2), kept="yes" if keep else "no, reverted"))
            if keep: kept, kept_rows, kept_dev = cfg, cur, d
        rows += [dict(r, cfg="final") for r in kept_rows]               # the config the loop ends with
        if rep == 1: say("Prompt written for the last kept config of repeat 1 (first 400 characters): " + kept["prompt"][:400])
    df = pd.DataFrame(rows); df.to_csv(f"{OUT_DIR}/agents_E_rows.csv", index=False); RES["E"] = df
    real = df[df.cfg != "final"]
    show(summ(real, ["cfg", "split"]), "E1. Success by config and split (percent, Wilson 95% interval), cost per ticket", "E_by_config")
    print("baseline, always the most common (queue, priority) pair: " + ", ".join(
        f"{s} {100 * maj([(x['queue'], x['priority']) for x in S]):.0f}%" for s, S in (("dev", DEV), ("test", TEST))))
    show(pd.DataFrame(log), "E2. What the loop kept (a change stays only if the dev score rises)", "E_loop")
    ea = real[(real.split == "dev") & (real.cfg.isin(["c0_baseline", "c1_check_step"]))].groupby(["rep", "cfg", "cat"]).size().unstack(fill_value=0).reset_index()
    show(ea, "E3. Error analysis on the dev set: what kind of failure each reply is (counts)", "E_errors")
    g = real.groupby(["rep", "cfg", "split"], observed=True).ok.agg(["sum", "count"]).unstack(); g.columns = [f"{a}_{b}" for a, b in g.columns]
    g = g.reset_index(); g["gap_points"] = (100 * (g["sum_dev"] / g["count_dev"] - g["sum_test"] / g["count_test"])).round(0)
    g["dev"] = [rate_ci(a, b) for a, b in zip(g.sum_dev, g.count_dev)]; g["test"] = [rate_ci(a, b) for a, b in zip(g.sum_test, g.count_test)]
    show(g[["rep", "cfg", "dev", "test", "gap_points"]], "E4. Dev against held-out (percent, Wilson 95% interval). gap_points = dev minus test. A positive gap means the change fits the dev set better than new tickets.", "E_gap")
    samples(real)
    ts = df[df.split == "test"]
    show(pd.DataFrame([paired(ts, ["rep", "id"], "cfg", "final", "c0_baseline")]), "E5. Held-out tickets: the final config against the baseline, exact McNemar", "E_paired")
    f, b = ts[ts.cfg == "final"], ts[ts.cfg == "c0_baseline"]
    say(f"Reading from the numbers: on held-out tickets the loop moved success from {100 * b.ok.mean():.0f}% to {100 * f.ok.mean():.0f}%; dev score rose from "
          f"{100 * real[(real.cfg == 'c0_baseline') & (real.split == 'dev')].ok.mean():.0f}% to {100 * df[(df.cfg == 'final') & (df.split == 'dev')].ok.mean():.0f}%.")

part("E", run_e)


## Part D: one agent or several?

Part D is the largest part and runs after part E, so that a call cap can only cut part D.

Question: for which tasks does a team of agents help, and what does it cost? The exam asks for multi-agent components, collaboration patterns, and when a single agent is the right choice.

Three task families: separable (three independent lookups), coupled (three lookups, each needs the result before it), and simple (one lookup). Four designs: one agent, a supervisor with parallel workers, a sequential pipeline (researcher, then writer), and a generator with a critic. Every task gets the same token budget, counted over all calls of the design. A design that spends the budget before it answers fails with the outcome `budget`. The rule that picks the single agent is fixed in the code before the run.


In [ ]:
# 13. Part D: one world, three task families, four designs, one token budget. Pure Python. check_D() tests the world and the grader.
rngd = random.Random(SEED + 3)
D_ITEMS = ["pump", "relay", "fan", "coil", "gear", "belt", "clamp", "motor", "valve", "pipe", "tank", "lens", "cable", "switch"]
D_PRICE, D_STOCK = {i: rngd.randint(15, 95) for i in D_ITEMS}, {i: rngd.randint(10, 99) for i in D_ITEMS}
WORKERS = ["ana", "ben", "cleo", "dev", "eli", "fay", "gus", "hana", "ivo", "jo", "kim", "lou"]
MANAGERS = ["raj", "sue", "tom", "uma", "vic", "wen", "xia", "yan", "zed", "abe", "bea", "cal"]
DEPTS = ["ops", "fin", "hr", "eng", "qa", "legal", "infra", "sales", "comms", "research", "lab", "facility", "security"]
D_DEPT = {d: dict(budget=10000 * (i + 3) + 100 * rngd.randint(1, 9), site=f"site{i}") for i, d in enumerate(DEPTS)}
D_EMP = {**{w: dict(dept=DEPTS[i], manager=MANAGERS[i]) for i, w in enumerate(WORKERS)}, **{m: dict(dept=DEPTS[i + 1], manager="board") for i, m in enumerate(MANAGERS)}}
D_REG = dict(get_employee=lambda name: D_EMP.get(name) or {"error": "unknown employee"}, get_dept=lambda dept: D_DEPT.get(dept) or {"error": "unknown dept"},
             get_price=lambda item: {"price": D_PRICE[item]} if item in D_PRICE else {"error": "unknown item"},
             get_stock=lambda item: {"stock": D_STOCK[item]} if item in D_STOCK else {"error": "unknown item"})
D_TOOLS = [tool("get_employee", "Get the department and the manager of an employee.", "name"), tool("get_dept", "Get the budget and the site of a department.", "dept"),
           tool("get_price", "Get the unit price of an item.", "item"), tool("get_stock", "Get the units in stock of an item.", "item")]
BUDGET_TOK = 4000      # tokens per task (prompt plus completion, over all calls). Every design gets the same cap.
D_TASKS = []
def addd(family, q, answer, typ, via): D_TASKS.append(dict(id=f"{family[:3]}{len(D_TASKS) % 12 + 1}", family=family, q=q, answer=answer, typ=typ, via=via))
for k in range(12):
    a = rngd.sample(D_ITEMS, 3)
    addd("separable", f"Report the stock level of {a[0]}, {a[1]} and {a[2]}. Reply with the three numbers separated by commas, in that order.",
         [D_STOCK[i] for i in a], "list", lambda a=a: [D_REG["get_stock"](i)["stock"] for i in a])
for w in WORKERS:                                         # three dependent lookups: employee, manager, the manager's department, its budget
    addd("coupled", f"What is the budget of the department that the manager of {w} belongs to? Reply with the number only.", D_DEPT[D_EMP[D_EMP[w]["manager"]]["dept"]]["budget"], "num",
         lambda w=w: D_REG["get_dept"](D_REG["get_employee"](D_REG["get_employee"](w)["manager"])["dept"])["budget"])
for i in rngd.sample(D_ITEMS, 12): addd("simple", f"What is the price of {i}? Reply with the number only.", D_PRICE[i], "num", lambda i=i: D_REG["get_price"](i)["price"])

def grade_d(task, text):
    text = text.split("</think>")[-1]
    if task["typ"] == "list":
        nums = re.findall(r"\d+", text); return [int(n) for n in nums] == task["answer"] if nums else None
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return abs(float(nums[-1]) - task["answer"]) < 1e-6 if nums else None

def check_D():
    assert grade_d(dict(typ="list", answer=[3, 14, 20]), "3, 14, 20") is True and grade_d(dict(typ="list", answer=[3, 14, 20]), "14, 3, 20") is False
    assert grade_d(dict(typ="num", answer=52000), "It is 52,000.") is True and grade_d(dict(typ="num", answer=52000), "51,000") is False and grade_d(dict(typ="num", answer=1), "none") is None
    assert Counter(t["family"] for t in D_TASKS) == dict(separable=12, coupled=12, simple=12)
    for t in D_TASKS: assert t["via"]() == t["answer"], t["id"]              # the simulator sets the truth: the tools give it by the path the agent must take
    for w in WORKERS: assert D_DEPT[D_EMP[w]["dept"]]["budget"] != D_DEPT[D_EMP[D_EMP[w]["manager"]]["dept"]]["budget"]   # the shortcut gives a wrong number
check_D()
print("part D world checked:", dict(Counter(t["family"] for t in D_TASKS)), "| token budget per task:", BUDGET_TOK)

# The four designs. Each takes a task and a Budget and returns outcome, answer and the calls and tokens it used.
D_SYS = "You answer questions with the tools. Use a tool for every fact. Reply with the final answer only, in the format the task asks."
def reply_outcome(r): return "error" if not r["ok"] else "cut" if r["finish"] == "length" else "answered" if r["content"].strip() else "empty"
def use(tot, r):
    tot["calls"] += r.get("calls", 1); tot["ptok"] += r["ptok"]; tot["ctok"] += r["ctok"]; tot["bad"] += r.get("bad", 0); return r
def ask(tot, bud, msgs, tag, max_tokens=100):          # one model call that counts against the budget
    r = chat(MODEL, msgs, tag, max_tokens=max_tokens); bud.add(r); return use(tot, r)
def agent(tot, bud, msgs, tag, max_steps=6): return use(tot, run_agent(msgs, D_TOOLS, D_REG, tag, bud, max_steps))

def d_single(task, bud):
    tot = dict(calls=0, ptok=0, ctok=0, bad=0); a = agent(tot, bud, [mk("system", D_SYS), mk("user", task["q"])], "D.single")
    return dict(tot, outcome=a["outcome"], answer=a["answer"])

def d_pipeline(task, bud):                             # sequential: a researcher with tools hands notes to a writer without tools
    tot = dict(calls=0, ptok=0, ctok=0, bad=0)
    a = agent(tot, bud, [mk("system", "You are a researcher. Use the tools to collect every fact the task needs. Reply with short notes of the facts only."), mk("user", task["q"])], "D.pipeline")
    if a["outcome"] != "answered": return dict(tot, outcome=a["outcome"], answer="")
    if bud.over(): return dict(tot, outcome="budget", answer="")
    r = ask(tot, bud, [mk("system", "Write the final answer from the notes. Reply with the final answer only, in the format the task asks."), mk("user", f"Task: {task['q']}\nNotes: {a['answer']}")], "D.pipeline")
    return dict(tot, outcome=reply_outcome(r), answer=r["content"].strip())

def d_supervisor(task, bud):                           # a planner splits the task, workers run at the same time, a synthesizer combines
    tot = dict(calls=0, ptok=0, ctok=0, bad=0)
    p = ask(tot, bud, [mk("system", "You are a supervisor. Split the task into at most 4 independent subtasks that different workers can do at the same time. If it cannot be split, "
                          "return one subtask with the whole task. Reply with JSON only: {\"subtasks\": [\"...\"]}"), mk("user", task["q"])], "D.supervisor", 200)
    if not p["ok"]: return dict(tot, outcome="error", answer="")
    subs = (jparse(p["content"]) or {}).get("subtasks")
    subs = [str(s) for s in subs][:4] if isinstance(subs, list) and subs else [task["q"]]
    wt = [dict(calls=0, ptok=0, ctok=0, bad=0) for _ in subs]
    with ThreadPoolExecutor(len(subs)) as ex:
        ws = list(ex.map(lambda i: agent(wt[i], bud, [mk("system", D_SYS), mk("user", subs[i])], "D.supervisor", 5), range(len(subs))))
    for w in wt: use(tot, w)
    if bud.over(): return dict(tot, outcome="budget", answer="")
    r = ask(tot, bud, [mk("system", "Combine the worker results into the final answer to the task. Reply with the final answer only, in the format the task asks."),
                       mk("user", f"Task: {task['q']}\n" + "\n".join(f"Subtask {i + 1}: {s}\nResult: {w['answer'] or w['outcome']}" for i, (s, w) in enumerate(zip(subs, ws))))], "D.supervisor")
    return dict(tot, outcome=reply_outcome(r), answer=r["content"].strip())

def d_gen_critic(task, bud):                           # a generator answers, a critic checks the answer against the tool results, one revision at most
    tot = dict(calls=0, ptok=0, ctok=0, bad=0); g = agent(tot, bud, [mk("system", D_SYS), mk("user", task["q"])], "D.gen_critic")
    if g["outcome"] != "answered": return dict(tot, outcome=g["outcome"], answer="")
    if bud.over(): return dict(tot, outcome="answered", answer=g["answer"])           # no budget left for the critic: the answer stands unchecked
    obs = "\n".join(m["content"] for m in g["msgs"] if m["role"] == "tool")
    c = ask(tot, bud, [mk("system", "You check an answer against the tool results. Reply with JSON only: {\"ok\": true} or {\"ok\": false, \"fix\": \"what to change\"}."),
                       mk("user", f"Task: {task['q']}\nTool results:\n{obs}\nAnswer: {g['answer']}")], "D.gen_critic")
    v = jparse(c["content"]) if c["ok"] else None
    if v is None or v.get("ok") is not False: return dict(tot, outcome="answered", answer=g["answer"])     # accepted, or the critic reply was not parsed
    if bud.over(): return dict(tot, outcome="budget", answer="")
    r = agent(tot, bud, g["msgs"] + [mk("user", f"A reviewer says: {v.get('fix', 'check the answer')}. Reply with the corrected final answer only.")], "D.gen_critic", 3)
    return dict(tot, outcome=r["outcome"], answer=r["answer"])

DESIGNS = dict(single=d_single, supervisor_parallel=d_supervisor, pipeline=d_pipeline, generator_critic=d_gen_critic)


In [ ]:
# 14. Part D: 36 tasks, four designs, once, one token budget per task. Tasks run one at a time, so wall time shows the design, not the queue.
TOL = 0.10    # the rule, fixed before the run: a single agent is the right choice for a family when it is within 10 points of the best design and uses fewer tokens than every other design

def run_d():
    dl = PART_END
    def one(job):
        rep, name, t = job; bud, t0 = Budget(BUDGET_TOK), time.time()
        res = DESIGNS[name](t, bud); v = grade_d(t, res["answer"]) if res["outcome"] == "answered" else None
        return fin(dict(rep=rep, family=t["family"], system=name, id=t["id"], outcome=agent_outcome(res, v), calls=res["calls"], ptok=res["ptok"], ctok=res["ctok"],
                        wall_s=time.time() - t0, tok_x=round((res["ptok"] + res["ctok"]) / BUDGET_TOK, 2), bad_tool_calls=res["bad"], reply=res["answer"][:120]))
    jobs = []
    for rep in range(1, REPS_BY["D"] + 1):
        js = [(rep, n, t) for n in DESIGNS for t in D_TASKS]; random.Random(SEED + rep).shuffle(js); jobs += js
    df = pd.DataFrame(rows_of(jobs, pmap(one, jobs, dl, workers=1), lambda j: dict(nr(rep=j[0], family=j[2]["family"], system=j[1], id=j[2]["id"]), tok_x=0.0, bad_tool_calls=0)))
    df.to_csv(f"{OUT_DIR}/agents_D_rows.csv", index=False); RES["D"] = df
    show(summ(df, ["family", "system"], means=("tok_x",)), f"D1. Success by task family and design (percent, Wilson 95% interval), cost per task. tok_x = tokens used / budget of {BUDGET_TOK}", "D_by_family")
    print("baseline, always the most common answer: " + ", ".join(f"{f} {100 * maj([x['answer'] for x in D_TASKS if x['family'] == f]):.0f}%" for f in ("separable", "coupled", "simple")))
    show(summ(df, ["system"]), "D2. All tasks pooled", "D_pooled")
    pr = [dict(family=f, **paired(d, ["rep", "id"], "system", "single", m)) for f, d in list(df.groupby("family")) + [("all", df)] for m in DESIGNS if m != "single"]
    show(pd.DataFrame(pr), "D3. Single against each multi-agent design, exact McNemar on the same task and repeat. only_a counts tasks only the single agent solved.", "D_paired")
    samples(df)
    print("\nDecision per family, from the numbers (rule: success within 10 points of the best design, and fewer tokens than every other design):")
    for f, d in df.groupby("family", sort=False):
        s, tk = d.groupby("system").ok.mean(), (d.ptok + d.ctok).groupby(d.system).mean()
        others = [m for m in DESIGNS if m != "single"]
        right = s["single"] >= s.max() - TOL and all(tk["single"] < tk[m] for m in others)
        print(f"  {f:10s} single {100 * s['single']:.0f}% at {tk['single']:.0f} tokens | best design {s.idxmax()} {100 * s.max():.0f}% | cheapest other {min(tk[m] for m in others):.0f} tokens"
              f" | single is {'the right choice' if right else 'not clearly the right choice'}")
    sp = df[df.system == "supervisor_parallel"].groupby("family").wall_s.mean(); si = df[df.system == "single"].groupby("family").wall_s.mean()
    print("Wall time, supervisor over single: " + ", ".join(f"{f} x{sp[f] / si[f]:.1f}" for f in sp.index) + " (above 1 means the supervisor is slower)")

part("D", run_d)


## Part F: NVIDIA tools for agents, run for real

Question: what are NeMo, the NeMo Agent Toolkit and NemoClaw for, and does the toolkit run here? The exam asks which NVIDIA tool builds, runs and secures agents.

The part prints what each tool is for, from the source files. Then it builds a `uv` environment with Python 3.12, installs `nvidia-nat` 1.9.0 with its LangChain, evaluation and profiler extras, and runs one ReAct workflow from a YAML file. It evaluates four questions with the `exact_match` evaluator, which needs no judge model, and it prints the profiler output. The part has a limit of 10 minutes. If a step fails, the real error is printed and the notebook goes on.


In [ ]:
# 15. Part F: NVIDIA NeMo Agent Toolkit (nvidia-nat 1.9.0) in its own uv venv, so it cannot change this kernel. One workflow: config, run, evaluate, profile. At most 10 minutes.
NAT_VERSION, NAT_LIMIT_S = "1.9.0", 600
TOOLS_TABLE = [
    ("NVIDIA NeMo", "A collection of tools to build specialized agent systems: data processing and generation, fine-tuning and evaluation, reinforcement learning, speech, safety and agent observability.", "ai-agents.txt"),
    ("NVIDIA NeMo Agent Toolkit", "An open source framework that wires the agent loop, works with LangChain, CrewAI and LlamaIndex, and profiles and optimizes agent systems (latency bottlenecks, token costs).",
     "agentic-inference.txt; building-nvidia-nemotron-3-agents-for-reasoning-multimodal-rag-voice-and-safety.txt"),
    ("NVIDIA NemoClaw", "An open source stack to run autonomous, long-running agents more securely: agents run in OpenShell sandboxes, with policy-enforced access to tools and data.", "ai-agents.txt")]
F_DATA = [dict(id="1", question="What is the current year? Reply with the four digit number only.", answer="2026"),      # needs the tool. True until 2026-12-31.
          dict(id="2", question="What is the capital of France? Reply with one word.", answer="Paris"),
          dict(id="3", question="How many days are in a leap year? Reply with the number only.", answer="366"),
          dict(id="4", question="Reply with the single word: ready", answer="ready")]
F_WORKFLOW = f"""functions:
  current_datetime:
    _type: current_datetime
llms:
  nim_llm:
    _type: nim
    model_name: {UP}
    max_tokens: 1024
workflow:
  _type: react_agent
  tool_names: [current_datetime]
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3
"""
def f_eval_section(out, data, profiler):
    prof = "    profiler:\n      compute_llm_metrics: true\n      csv_exclude_io_text: true\n      bottleneck_analysis:\n        enable_nested_stack: true\n" if profiler else ""
    return (f"eval:\n  general:\n    output_dir: {out}\n    max_concurrency: 2\n    dataset:\n      _type: json\n      file_path: {data}\n{prof}"
            "  evaluators:\n    exact:\n      _type: langsmith\n      evaluator: exact_match\n")

def sh(cmd, timeout, env=None):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=env)
        return p.returncode, scrub(p.stdout + p.stderr)
    except subprocess.TimeoutExpired:
        return 124, f"timeout after {timeout:.0f}s"
    except OSError as e:
        return 127, scrub(e)
def tail(text, n=22): return "\n".join(l[:190] for l in text.strip().splitlines()[-n:])

def run_f():
    print("F1. What each NVIDIA tool is for (paraphrased from the named source files in hub/sources)")
    for name, what, src in TOOLS_TABLE: print(textwrap.fill(f"{name}: {what} [{src}]", 185, subsequent_indent="    "))
    pd.DataFrame(TOOLS_TABLE, columns=["tool", "what_it_is_for", "source"]).to_csv(f"{OUT_DIR}/agents_F_tools.csv", index=False)
    end = time.time() + min(NAT_LIMIT_S, PART_END - time.time())
    d = os.path.abspath(f"{OUT_DIR}/nat"); os.makedirs(d, exist_ok=True)
    py, nat, out, data, cfg, ecfg = f"{d}/venv/bin/python", f"{d}/venv/bin/nat", f"{d}/out", f"{d}/dataset.json", f"{d}/config.yml", f"{d}/eval_config.yml"
    env, log = dict(os.environ, NVIDIA_API_KEY=KEY), []
    def step(name, cmd, cap, tolerate=False):
        t = time.time(); rc, o = sh(cmd, max(5, min(cap, end - t)), env)
        log.append(dict(step=name, rc=rc, seconds=round(time.time() - t))); print(f"\n[{name}] rc={rc}\n" + tail(o))
        if rc and not tolerate: raise RuntimeError(f"{name} failed with rc={rc}. The real error is printed above.")
        return rc
    step("uv venv python 3.12", ["uv", "venv", "--python", "3.12", f"{d}/venv"], 120)
    step("install nvidia-nat", ["uv", "pip", "install", "--python", py, f"nvidia-nat[langchain,eval,profiler]=={NAT_VERSION}"], 360)
    json.dump(F_DATA, open(data, "w")); open(cfg, "w").write(F_WORKFLOW)
    step("nat run", [nat, "run", "--config_file", cfg, "--input", F_DATA[0]["question"]], 120)
    for prof in (True, False):                          # if the profiler section is the problem, try once more without it
        open(ecfg, "w").write(F_WORKFLOW + f_eval_section(out, data, prof))
        if step("nat eval" + (" with profiler" if prof else " without profiler"), [nat, "eval", "--config_file", ecfg], 240, tolerate=prof) == 0: break
    files = sorted(os.listdir(out)) if os.path.isdir(out) else []
    print("\noutput files:", files)
    if "workflow_output.json" in files:
        for r in json.load(open(f"{out}/workflow_output.json")): print(f"  question {str(r.get('question'))[:48]!r} expected {r.get('answer')!r} got {str(r.get('generated_answer'))[:40]!r}")
    for f in files:
        if f.endswith("_output.json") and f != "workflow_output.json":
            j = json.load(open(f"{out}/{f}")); print(f, {k: v for k, v in j.items() if not isinstance(v, (list, dict))})
    if "standardized_data_all.csv" in files:
        p = pd.read_csv(f"{out}/standardized_data_all.csv"); print(f"\nprofiler table: {len(p)} rows, columns {list(p.columns)[:10]}")
        if "event_type" in p: print("events:", p.event_type.value_counts().to_dict())
        print(p.select_dtypes("number").describe().loc[["count", "mean", "max"]].round(2).iloc[:, :8].to_string())
    if "workflow_profiling_report.txt" in files: print("\n" + tail(open(f"{out}/workflow_profiling_report.txt").read(), 30))
    show(pd.DataFrame(log), "F2. Steps, exit codes and seconds", "F_steps")

part("F", run_f)


In [ ]:
# 16. Last cell: one figure, the status of every part, the ledger of every call and token (from the API usage field), cleanup.
def plot():
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    if "B" in RES:
        for var, m in (("clean", "o"), ("noisy", "s")):
            d = RES["B"][RES["B"]["var"] == var].groupby("strat").agg(k=("e_hits", "sum"), n=("e_items", "sum"), tok=("ans_ptok", "mean"))
            ax[0].scatter(d.tok, 100 * d.k / d.n, marker=m, label=var)
            for s, r in d.iterrows(): ax[0].annotate(s, (r.tok, 100 * r.k / r.n), fontsize=7)
        ax[0].set(xlabel="prompt tokens of the answer call", ylabel="early facts right (%)", title="B: accuracy against prompt tokens"); ax[0].legend()
    if "D" in RES:
        RES["D"].groupby(["family", "system"]).ok.mean().unstack().mul(100).reindex(ORDER["family"])[ORDER["system"]].plot.bar(ax=ax[1], rot=0, width=0.8)
        ax[1].set(ylabel="success (%)", title="D: success by family and design"); ax[1].legend(fontsize=6)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/agents_summary.png", dpi=110); plt.close(fig)
try: plot()
except Exception as e: print("plot failed:", scrub(repr(e))[:150])

print(pd.DataFrame(STATUS).to_string(index=False)); print(f"\nwhole notebook: {time.time() - T0:.0f} s")
L = pd.DataFrame(LEDGER, columns=["part", "tag", "model", "status", "ptok", "ctok", "lat"]); L.to_csv(f"{OUT_DIR}/agents_call_ledger.csv", index=False)
assert len(L) == STATE["attempts"], "the ledger and the attempt counter must agree"
agg = dict(attempts=("status", "size"), http_200=("status", lambda s: int((s == 200).sum())), http_429=("status", lambda s: int((s == 429).sum())),
           failed_other=("status", lambda s: int(((s != 200) & (s != 429)).sum())), prompt_tok=("ptok", "sum"), completion_tok=("ctok", "sum"))
by_part = L.groupby("part").agg(**agg); by_part.loc["TOTAL"] = by_part.sum()
print("\nCalls and tokens by part. Every number counts HTTP attempts, retries included, and tokens from the usage field:"); print(by_part.to_string())
print("\nHTTP status of every attempt, by model (200 = ok, 429 = rate limit, 0 = network error or timeout). A failed call is never only 'error':")
print(L.groupby(["model", "status"]).size().unstack(fill_value=0).to_string())
print("\nCalls and tokens by call site:"); print(L.groupby("tag").agg(**agg).to_string())
print(f"\ncall cap {CALL_CAP}: used {STATE['attempts']}. Part F calls made by nvidia-nat run in a subprocess and are not in this ledger.")
rc, o = sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], 20)
print("servers started by this notebook: none. GPU memory in use:", o.strip().replace("\n", " | ") if rc == 0 else "no GPU in this CPU notebook")


## What to look for

- **Part A0.** A model that does not call a tool cannot pass a tool task. Read `tool_%` next to `success`. Reasoning off can make a model answer from its own words ("I need the order ID") instead of calling the tool. The seconds include the pacing wait.
- **Part A.** Read `success`, then `calls` and `tok`. Look at the `other` column: `no_rule` means no rule matched, `wrong` means a rule matched with an old value. Compare the kinds in A2. One task is about 4 points of 24, and the interval shows how much a rate can move.
- **Part B.** Compare `early` and `late` for each strategy. Stateless is the floor. The sliding window should keep only recent facts. Compare `ans_ptok` (prompt tokens of the answer call) with the accuracy. `rot_points` is the early-fact loss from noise. A strong model may show little rot. The decoys are simple, and one run has only 20 early facts per strategy.
- **Part C.** Compare the stores on the cross-session questions. `stale` is the old city. `leak` is another user's value. C3 shows the store without a model: whether the right sentence was found and whether the profile holds the true fact. Five users give 10 cross-session questions per store and repeat.
- **Part D.** Compare `success` inside one family. `tok_x` is tokens used divided by the budget. `budget` in `other` means the design ran out of tokens. Wall time is mean seconds per task. Tasks run one at a time, so the time shows the design and not a queue. There are 12 tasks per family and one run. The decision lines come from the printed numbers and the rule in the code.
- **Part E.** Compare dev and test. A large positive `gap_points` means the change fits the dev tickets and not new ones. The error analysis model reads the dev labels. Held-out tickets of the kinds `hack` and `general` repeat a few sentences from the dev set. One ticket is about 5 points of 20.
- **Part F.** If the install worked, read the evaluator average, the answers next to the expected text, and the profiler rows. `exact_match` is strict: an answer with a full stop fails. If a step failed, read the error text above.
- **All parts.** The API is not deterministic, so compare the two repeats (`_r1`, `_r2`) before you trust a difference. Repeats of one item are not independent. The p values are a guide, not a proof. A cut, empty, unparsed, error or not-reached reply is an outcome of its own and never leaves the table.


## Exam rules this proves

1. **agentic.1**: A generative AI call turns one prompt into one reply. An agent adds tools, memory, planning and a loop, so it can fetch live facts and take several steps. A rule-based system answers only what its author wrote down, and its data goes stale when the world changes.
2. **agentic.1**: The building blocks of an agent are the model, tools, memory, planning, a harness and a runtime (`ai-agents.txt`). The harness is the scaffolding that connects memory and tools to the model. The runtime is where the agent runs, with limits.
3. **agentic.10**: A stateless call sees only the current request. A stateful interaction keeps state between turns. Context engineering chooses what enters the window. A short state object can hold the facts that matter at a fraction of the tokens. Long noisy history costs tokens and can lose facts.
4. **agentic.14**: Short-term memory is the current session and lives in the context. Long-term memory lives outside the context and lasts across sessions (`introduction-to-llm-agents.txt`, `ai-agents.txt`). Use short-term memory for the live task. Use long-term memory for facts that must outlive a session.
5. **agentic.14**: A long-term store needs a user scope, an overwrite rule for facts that change, and a way to find the right item. Without them it gives stale or foreign facts.
6. **agentic.11**: A multi-agent system has agents with roles, an orchestrator, shared state and handoffs (`multi-agent-systems.txt`). Common patterns are supervisor, sequential pipeline, and generator with critic. They cost more calls and tokens. Choose a single agent when the task is simple or tightly coupled and the numbers show no gain. Choose several agents when the parts are independent and time matters.
7. **agentic.4**: Improve an agent in a loop. Collect feedback, read the errors, change one thing at a time (prompt, workflow, model), keep a change only when the dev score rises, and measure on held-out items.
8. **agentic.8**: NVIDIA NeMo is a set of tools for building agent systems. The NeMo Agent Toolkit wires the agent loop from a YAML file and profiles and evaluates it. NemoClaw runs long-running agents in a sandbox with policy-enforced access.

| Objective | Exam term | Plain line | Shown by |
|---|---|---|---|
| agentic.1 | Agentic AI concepts | An agent pursues a goal with tools in a loop. | Part A |
| agentic.1 | Agents versus generative AI | One call answers once. An agent acts, reads and acts again. | A1: one call against agent |
| agentic.1 | Agents versus rule-based systems | Rules match fixed patterns and use data from design time. | A2: no_rule and stale rules |
| agentic.1 | Core building blocks | Model, tools, memory, planning, harness, runtime. | A: building block table |
| agentic.1 | Harnesses | The code that runs tool calls and stops at limits. | `run_agent()` and its limits |
| agentic.1 | Runtimes | The place that runs the agent under limits. | `pmap`, call cap, deadlines |
| agentic.4 | Feedback | Measured errors drive the next change. | E3: error analysis |
| agentic.4 | Prompt refinement | A better instruction, written from the errors. | E: config c2 |
| agentic.4 | Workflow refinement | An added check step with a retry. | E: config c1 |
| agentic.4 | Model selection | A stronger or cheaper model for the job. | A0 and E: config c3 |
| agentic.8 | NVIDIA NeMo | A collection of tools to build agent systems. | F1 table |
| agentic.8 | NVIDIA Agent Toolkit | An open source framework to wire, profile and evaluate agents. | F: `nat run`, `nat eval` |
| agentic.8 | NVIDIA NemoClaw | A stack for long-running agents in sandboxes. | F1 table |
| agentic.8 | NVIDIA tools for building agents | NeMo, the Agent Toolkit and NemoClaw together. | F1 table |
| agentic.10 | Stateless interactions | Each request stands alone. | B: stateless |
| agentic.10 | Stateful interactions | The system keeps state between turns. | B: full history, state object |
| agentic.10 | Context engineering | Choose what goes into the window. | B1: tokens against accuracy |
| agentic.10 | State and context management | Keep the needed facts, drop the rest. | B2: context rot |
| agentic.11 | Multi-agent components | Roles, an orchestrator, shared state, handoffs. | D: four designs |
| agentic.11 | Collaboration patterns | Supervisor, pipeline, generator and critic. | D1 |
| agentic.11 | Single-agent versus multi-agent design | Compare success, tokens and time at equal budget. | D1, D3 |
| agentic.11 | When to choose single-agent design | Simple or tightly coupled tasks. | D decision lines |
| agentic.14 | Short-term memory | The current session, held in the context. | C: store none, same question |
| agentic.14 | Long-term memory | A store that lasts across sessions. | C: log, BM25, profile |
| agentic.14 | When each memory type is appropriate | Short for the live task, long for facts across sessions. | C1, C2 |
